# Entrenamiento de la U-Net en Kaggle (BBBC039)

Entrena `scripts/train.py` del repositorio en GPU y deja los pesos para descargar.

**Antes de ejecutar** (panel derecho → *Settings*):
- *Accelerator*: **GPU** (T4 o P100).
- *Internet*: **On** (hace falta para clonar el repo y descargar BBBC039).

Todo lo que se escribe queda en `/kaggle/working/`, que es lo que Kaggle conserva y deja
descargar al guardar la versión (*Save Version → Save & Run All*).

In [ ]:
!nvidia-smi

## 1. Clonar el repositorio e instalar el paquete

In [ ]:
%cd /kaggle/working
!rm -rf repo
!git clone --depth 1 https://github.com/Jesus-OP/nuclei-segmentation-demo.git repo
%cd /kaggle/working/repo
!git log -1 --format='commit %h  %cd'
!pip install -q -e .

## 2. Descargar BBBC039 (imágenes, máscaras y splits oficiales, licencia CC0)

In [ ]:
!python scripts/download_data.py --out /kaggle/working/data

## 3. Entrenar

Selección del mejor modelo por AP@0.5:0.95 en val (test no se usa). Semilla fija.
Con los valores por defecto (150 épocas) se estima < 1 h en una T4 (no medido); para una prueba
rápida, añadir `--epochs 5`.

In [ ]:
!python scripts/train.py \
    --data /kaggle/working/data \
    --out /kaggle/working/runs/unet \
    --workers 4

## 4. Curva de entrenamiento

In [ ]:
import pandas as pd
log = pd.read_csv("/kaggle/working/runs/unet/log.csv")
ax = log.plot(x="epoch", y="train_loss", figsize=(8, 3), title="train loss")
v = log.dropna(subset=["val_AP@0.5:0.95"])
v.plot(x="epoch", y=["val_AP@0.5", "val_AP@0.5:0.95"], figsize=(8, 3), title="val")
v.tail()

## 5. Empaquetar pesos y log para descargar

Descargar `unet_weights.zip` (enlace abajo o pestaña *Output*) y descomprimirlo en
`runs/unet/` del repo local. Después, en local (CPU):

```
python scripts/predict_unet.py --weights runs/unet/best.pt --split test
python scripts/evaluate.py
python scripts/report.py
```

In [ ]:
import shutil
from IPython.display import FileLink
%cd /kaggle/working
shutil.make_archive("unet_weights", "zip", "runs/unet", ".")
FileLink("unet_weights.zip")